<div style="padding:2.2rem;border-radius:18px;background:linear-gradient(135deg,#1e3a8a,#0f766e);color:white;">
<p style="font-size:1.05rem;letter-spacing:.12em;text-transform:uppercase;margin:0;">D150 · Database Design</p>
<h1 style="font-size:3rem;margin:.5rem 0;">Normalization Basics</h1>
<p style="font-size:1.35rem;margin:0;">Organizing e-commerce data so it stays accurate, consistent, and easy to maintain</p>
</div>

### Running example: **QuickCart**
We will improve one e-commerce order table step by step—without writing SQL.

# Learning outcomes

By the end, you should be able to:

- Explain **why** normalization is useful
- Recognize repeating groups and duplicated facts
- Describe **First, Second, and Third Normal Form**
- Split one large table into related tables
- Identify primary keys and foreign keys
- Know when a small amount of duplication may be intentional

# Why normalize data?

**Normalization** is a method of organizing relational data so that each fact is stored in the right place, usually only once.

Without normalization, QuickCart may face:

| Problem | E-commerce consequence |
|---|---|
| Repeated data | Customer address is copied into every order line |
| Inconsistent data | The same product has two different names |
| Difficult updates | A price or email must be changed in many rows |
| Accidental data loss | Deleting an order removes the only product record |
| Wasted storage | The same descriptions appear thousands of times |

# QuickCart's unnormalized order sheet

QuickCart initially stores one row per order, with several products packed into a single cell.

| Order | Date | Customer | Email | City | Products | Quantities |
|---|---|---|---|---|---|---|
| O101 | 2026-08-01 | venkat | venkat@example.com | Chennai | P10:Mouse, P20:Keyboard | 2, 1 |
| O102 | 2026-08-02 | venkat | venkat@example.com | Chennai | P30:Webcam | 1 |

### What is wrong?

- `Products` and `Quantities` contain **lists**, not single values.
- Matching each quantity to its product depends on position.
- Searching, sorting, or validating one product is difficult.
- Customer facts are repeated for every order.

# Before the normal forms: keys and dependencies

A **primary key** uniquely identifies a row. A **foreign key** connects a row to another table.

A **functional dependency** means one value determines another:

| Dependency | Meaning |
|---|---|
| `CustomerID → CustomerName, Email, City` | One customer ID identifies the customer's details |
| `ProductID → ProductName, CurrentPrice` | One product ID identifies the product's details |
| `OrderID → OrderDate, CustomerID` | One order identifies its date and customer |
| `(OrderID, ProductID) → Quantity, SalePrice` | One order-product pair identifies line details |

> These business rules guide where each column belongs.

# First Normal Form (1NF)

A table is in **1NF** when:

- Every cell holds **one value**
- There are no repeating groups or numbered columns
- Each row can be uniquely identified

We create one row for each product in an order:

| OrderID | ProductID | ProductName | Quantity | CustomerID | CustomerName | City |
|---|---|---|---:|---|---|---|
| O101 | P10 | Mouse | 2 | C01 | venkat | Chennai |
| O101 | P20 | Keyboard | 1 | C01 | venkat | Chennai |
| O102 | P30 | Webcam | 1 | C01 | venkat | Chennai |

**Key:** `(OrderID, ProductID)` uniquely identifies each order line.

# 1NF helps—but duplication remains

The table has atomic values, yet it still mixes facts about **orders, customers, products, and order lines**.

| Repeated fact | Why it repeats |
|---|---|
| Order date and customer ID | Once for every product in the order |
| Product name | Once for every order containing that product |
| Customer name and city | Once for every item the customer orders |

This creates three common anomalies:

- **Update:** changing venkat's city requires many edits.
- **Insert:** a new product cannot be recorded until someone orders it.
- **Delete:** deleting the last order for a product may erase its details.

# Second Normal Form (2NF)

A table is in **2NF** when it is in 1NF and every non-key column depends on the **whole** primary key.

Our order-line key is `(OrderID, ProductID)`, but:

- `OrderDate` and `CustomerID` depend only on `OrderID`.
- `ProductName` and `CurrentPrice` depend only on `ProductID`.
- `Quantity` and `SalePrice` depend on the complete pair.

These are **partial dependencies**. Move each fact to the table whose key fully determines it.

> 2NF matters mainly when a table has a composite key.

# QuickCart after 2NF

### Orders
| OrderID (PK) | OrderDate | CustomerID | CustomerName | Email | City |
|---|---|---|---|---|---|
| O101 | 2026-08-01 | C01 | venkat | venkat@example.com | Chennai |
| O102 | 2026-08-02 | C01 | venkat | venkat@example.com | Chennai |

### Products
| ProductID (PK) | ProductName | CurrentPrice |
|---|---|---:|
| P10 | Mouse | 500 |
| P20 | Keyboard | 1200 |
| P30 | Webcam | 2400 |

### OrderItems
| OrderID (PK, FK) | ProductID (PK, FK) | Quantity | SalePrice |
|---|---|---:|---:|
| O101 | P10 | 2 | 450 |
| O101 | P20 | 1 | 1200 |
| O102 | P30 | 1 | 2300 |

# Third Normal Form (3NF)

A table is in **3NF** when it is in 2NF and non-key columns do not depend on other non-key columns.

In `Orders`:

```text
OrderID → CustomerID → CustomerName, Email, City
```

The customer details depend on `CustomerID`, not directly on `OrderID`. This indirect relationship is a **transitive dependency**.

### Fix

Store customer details once in `Customers`. Keep only `CustomerID` in `Orders` as a foreign key.

# Final 3NF design

### Customers
| CustomerID (PK) | CustomerName | Email | City |
|---|---|---|---|
| C01 | venkat | venkat@example.com | Chennai |

### Orders
| OrderID (PK) | OrderDate | CustomerID (FK) |
|---|---|---|
| O101 | 2026-08-01 | C01 |
| O102 | 2026-08-02 | C01 |

### Products
| ProductID (PK) | ProductName | CurrentPrice |
|---|---|---:|
| P10 | Mouse | 500 |
| P20 | Keyboard | 1200 |
| P30 | Webcam | 2400 |

### OrderItems
| OrderID (PK, FK) | ProductID (PK, FK) | Quantity | SalePrice |
|---|---|---:|---:|
| O101 | P10 | 2 | 450 |
| O101 | P20 | 1 | 1200 |
| O102 | P30 | 1 | 2300 |

# How the tables work together

```text
Customers                         Products
 CustomerID (PK)                   ProductID (PK)
       │                                 │
       │ 1                               │ 1
       ▼ many                            ▼ many
Orders ─────────────────────────── OrderItems
 OrderID (PK)      1          many  OrderID (PK, FK)
 CustomerID (FK)                     ProductID (PK, FK)
                                     Quantity, SalePrice
```

- One customer can place many orders.
- One order can contain many order items.
- One product can appear in many order items.
- `OrderItems` resolves the many-to-many relationship between orders and products.

# What normalization fixes

| Earlier problem | Normalized solution |
|---|---|
| venkat's details repeated on every item | Stored once in `Customers` |
| Product name repeated across orders | Stored once in `Products` |
| Products packed into one cell | One row per item in `OrderItems` |
| Product cannot exist without an order | Product can be added independently |
| Deleting an order loses product details | Product remains in `Products` |
| Conflicting customer or product values | One authoritative row for each entity |

### Important detail
`SalePrice` belongs in `OrderItems` because it records the price actually charged. `CurrentPrice` belongs in `Products` and may change later.

# Normalization is a design tool, not a contest

For transactional systems, **3NF is a practical target** because it balances integrity and usability.

Sometimes teams intentionally duplicate selected data for faster reporting. This is called **denormalization**.

| Normalize when... | Consider denormalizing when... |
|---|---|
| Data changes frequently | Reads greatly outnumber writes |
| Accuracy and consistency are critical | A measured performance problem exists |
| The system processes transactions | Data is prepared for analytics/reporting |

> Start with a clear normalized design. Duplicate data only for a documented reason, with a plan to keep it consistent.

# Recap: the normalization journey

| Stage | Main question | QuickCart action |
|---|---|---|
| Unnormalized | Are values grouped or repeated? | Products were stored as lists |
| **1NF** | Does every cell contain one value? | Created one row per order item |
| **2NF** | Does every column depend on the whole key? | Separated orders and products from order items |
| **3NF** | Do non-key columns depend only on the key? | Moved customer details into `Customers` |

### Final idea

A well-normalized design gives each fact a natural home:

**Customers describe people · Products describe items · Orders describe purchases · OrderItems describe what was purchased**